# 🎬 Otaku AI Viral Shorts Studio — Kaggle Edition
Runs on **Kaggle T4 / P100 GPU** (30 hours/week free GPU). Supports scheduled autonomous execution without keeping browser tabs open.

### Setup:
1. Enable **Settings -> Accelerator -> GPU T4 x2** (or P100)
2. Enable **Settings -> Internet -> On**
3. (Optional) In **Add-ons -> Secrets**, add:
   - `GEMINI_API_KEY`: Your Gemini 2.5 Flash API Key
   - `YOUTUBE_COOKIES_TXT`: Netscape format cookies text for YouTube
4. Click **Run All** (or schedule daily via Kaggle Schedule).

In [ ]:
# ============================================================
# CONFIGURATION PARAMETERS
# ============================================================
RUN_FULL_CYCLE = True       # Runs all 4 categories sequentially
PILLAR_DAY_OVERRIDE = 0     # 0=Auto wheel | 1=Cinema | 2=TV Series | 3=The Mentalist | 4=Song Trends
TOPIC_OVERRIDE = ""         # Set to force topic, or leave blank for dynamic virality scan
USE_LOCAL_SMALL_MODEL = False # True=Ollama vision, False=Gemini Flash (fastest)
DURATION_MODE = "AI Decides Cut Duration (Full Climax)"
CUSTOM_SECONDS = 18
UPLOAD_TO_YOUTUBE = False
GEMINI_API_KEY = ""

import os, sys, subprocess, shutil, importlib, pathlib, time
print("⚙️ [1/4] Preparing Kaggle Environment...")

# 1. Environment and dependencies
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"
os.environ["TRANSFORMERS_TRUST_REMOTE_CODE"] = "1"

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get update -qq 2>/dev/null && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation >/dev/null 2>&1", shell=True)

_pkgs = [
    "yt-dlp", "pillow", "opencv-python-headless", "google-generativeai",
    "google-api-python-client", "google-auth-oauthlib", "numpy",
    "git+https://github.com/Panniantong/Agent-Reach.git",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"] + _pkgs, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 2. Clone or update repository in working directory
print("📦 [2/4] Syncing studio repository...")
_REPO_URL = "https://github.com/sp25126/MOVIE-SERIES---AUTO.git"
_REPO_DIR = "/kaggle/working/otaku_repo" if os.path.exists("/kaggle/working") else "otaku_repo"

if os.path.isfile("autonomous_pipeline.py"):
    subprocess.run(["git", "fetch", "-q", "origin", "main"], check=False)
    subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], check=False)
else:
    if not os.path.isdir(_REPO_DIR):
        subprocess.run(["git", "clone", "-q", _REPO_URL, _REPO_DIR], check=True)
    os.chdir(_REPO_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# 3. Load Kaggle / Environment Secrets
print("🔑 [3/4] Loading API keys and credentials...")
os.makedirs("config", exist_ok=True)
if not GEMINI_API_KEY.strip():
    try:
        from kaggle_secrets import UserSecretsClient
        GEMINI_API_KEY = (UserSecretsClient().get_secret("GEMINI_API_KEY") or "").strip()
    except Exception:
        pass
if not GEMINI_API_KEY.strip() and os.path.exists(".env"):
    for l in pathlib.Path(".env").read_text().splitlines():
        if l.strip().startswith("GEMINI_API_KEY="):
            GEMINI_API_KEY = l.split("=", 1)[1].strip().strip('"').strip("'")
            break
if GEMINI_API_KEY.strip():
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY.strip()
    print(f"   ✅ Gemini API key loaded ({len(GEMINI_API_KEY)} chars)")

# YouTube Cookies & OAuth Token for Scheduled Uploads
try:
    from kaggle_secrets import UserSecretsClient
    _ck = UserSecretsClient().get_secret("YOUTUBE_COOKIES_TXT") or ""
    if _ck.strip():
        pathlib.Path("config/cookies.txt").write_text(_ck, encoding="utf-8")
        print("   ✅ YouTube cookies loaded from Kaggle Secrets.")
    _tok = UserSecretsClient().get_secret("YOUTUBE_TOKEN_JSON") or ""
    if _tok.strip():
        pathlib.Path("config/youtube_token.json").write_text(_tok, encoding="utf-8")
        print("   ✅ YouTube OAuth token loaded from Kaggle Secrets.")
except Exception:
    pass

# 4. Reload modules and run pipeline
print("🚀 [4/4] Executing autonomous viral shorts production...")
import category_skills, proxy_harvester, clip_harvester, channel_intelligence, llm_director, dsp_renderer
import metadata_uploader, autonomous_pipeline
for _m in (category_skills, proxy_harvester, clip_harvester, channel_intelligence, llm_director, dsp_renderer, metadata_uploader, autonomous_pipeline):
    try:
        importlib.reload(_m)
    except Exception:
        pass

CLIP_DUR = 0.0 if ("AI Decides" in DURATION_MODE or "Full Uncut" in DURATION_MODE) else float(CUSTOM_SECONDS)

if RUN_FULL_CYCLE and not TOPIC_OVERRIDE.strip():
    _results = autonomous_pipeline.run_full_category_cycle(
        use_local_vlm=bool(USE_LOCAL_SMALL_MODEL),
        upload=bool(UPLOAD_TO_YOUTUBE),
        clip_dur=CLIP_DUR,
        start_day=int(PILLAR_DAY_OVERRIDE) if int(PILLAR_DAY_OVERRIDE) in (1, 2, 3, 4) else 1,
    )
else:
    _results = [autonomous_pipeline.run_autonomous_workflow(
        day_override=int(PILLAR_DAY_OVERRIDE),
        topic_override=TOPIC_OVERRIDE.strip(),
        use_local_vlm=bool(USE_LOCAL_SMALL_MODEL),
        upload=bool(UPLOAD_TO_YOUTUBE),
        clip_dur=CLIP_DUR,
    )]

for r in _results:
    if r.get("status") == "success":
        meta = r["metadata"]
        p = pathlib.Path(r["output_mp4"])
        print(f"\n{'='*60}")
        print(f"🎬 DONE: {p.name} ({p.stat().st_size / 1e6:.2f} MB)")
        print(f"Category: {r['category']} | Topic: {r['topic']}")
        print(f"Title:    {meta['title']}")
        print(f"Slot UTC: {meta['recommended_slot_utc']}")
        print(f"Tags:     {' '.join(meta.get('tags', [])[:10])}")
        print(f"Description Preview:\n{meta['description'][:250]}...")
